### ЗАДАЧА: Пакетная загрузка конфигов деплоя

От DevOps-команды приходит пакет строк с конфигами сервисов для выкладки.
Нужно обработать их так, чтобы:
- валидные конфиги попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, какие сервисы включены по окружениям и какой у них средний timeout.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестное окружение или неправильный флаг включения.


In [ ]:

# service|max_retries|timeout_sec|environment|enabled
rows = [
    'auth|3|1.5|prod|on',
    'billing|0|2.0|stage|on',
    'search|two|0.8|dev|off',
    'media|5|-1|prod|on',
    'chat|4|1.2|test|off',
    'mail|2|0.5|stage|maybe',
    'worker|1|3.4|prod|on',
]


class DeployConfigError(Exception):
    pass


class RowFormatError(DeployConfigError):
    pass


class RetriesError(DeployConfigError):
    pass


class TimeoutError(DeployConfigError):
    pass


class EnvironmentError(DeployConfigError):
    pass


class EnabledFlagError(DeployConfigError):
    pass


def parse_config(row):
    # TODO: распарсить строку и провалидировать max_retries, timeout_sec, environment, enabled
    parts = row.split('|')
    if len(parts) != 5:
        raise RowFormatError("Неверный формат записи")
    service,max_retries,timeout_sec,environment,enabled = parts
    if enabled == "on":
        enabled = True
    else:
        enabled = False
        raise RetriesError("Ошибка закрыто")
    # TODO: при ошибках конвертации использовать raise ... from ...
    try:
        max_retries= int(max_retries)
    except ValueError as e:
        raise RetriesError("попытки должны быть числом")
    
    if max_retries < 0:
        raise RetriesError("попытки должны быть положительны")
    
    try: 
        timeout_sec = float(timeout_sec)
    except ValueError as e:
        raise TimeoutError("время должно быть числом")
    
    if timeout_sec < 0:
        raise TimeoutError("время не может быть отрицательно")
    
    alloved_environment= {"prod","stage","dev"}
    if environment not in alloved_environment:
        raise EnvironmentError("недопустимая среда")
    return {
            'service': service,
            'max_retries': max_retries,
            'timeout_sec': timeout_sec,
            'environment': environment,
            'enabled': enabled
        }

    # TODO: enabled вернуть как bool

def load_configs(rows):
    # TODO: вернуть (configs, errors)
    configs = []
    errors = []

    for row in rows:
        try:
            configs.append(parse_config(row))
        except DeployConfigError as e:
            errors.append((row, type(e).__name__, str(e)))
    return configs, errors

# TODO: вызвать load_configs(rows)

configs, errors = load_configs(rows)

# TODO: вывести число валидных конфигов и число ошибок
print(f"Валидных конфигов: {len(configs)}")
print(f"Ошибок: {len(errors)}")

# TODO: вывести ошибки по типам

# TODO: собрать enabled_by_environment: dict[str, list[str]]

# TODO: посчитать average_timeout только по enabled=True
print("\nОшибки по типам:")
error_type_count = {}
for error in errors:
    error_type = error[1]  # тип ошибки из кортежа
    error_type_count[error_type] = error_type_count.get(error_type, 0) + 1

for error_type, count in error_type_count.items():
    print(f"  {error_type}: {count}")

# Собираем enabled_by_environment: dict[str, list[str]]
enabled_by_environment = {}
for config in configs:
    if config['enabled']:
        env = config['environment']
        if env not in enabled_by_environment:
            enabled_by_environment[env] = []
        enabled_by_environment[env].append(config['service'])

print("\nВключённые сервисы по окружениям:")
for env, services in enabled_by_environment.items():
    print(f"  {env}: {services}")

# Считаем average_timeout только по enabled=True
enabled_timeouts = [config['timeout_sec'] for config in configs if config['enabled']]
if enabled_timeouts:
    average_timeout = sum(enabled_timeouts) / len(enabled_timeouts)
    print(f"\nСредний timeout для включённых сервисов: {average_timeout:.2f} сек")
else:
    print("\nНет включённых сервисов для расчёта среднего timeout")
